# 🎲 Notebook 13: Monte Carlo Validation of Statistical Procedures

> Run with `jupyter nbconvert --to notebook --execute <this notebook>` (or open it in JupyterLab) to populate outputs. Every random step is seeded, so results are reproducible.

## Aims

A simulation study is the standard way to check whether a statistical method actually
delivers what it promises (unbiasedness, nominal coverage, correct size, advertised power).
This notebook follows the **ADEMP** structure of Morris, White & Crowther (2019) and reports
every performance measure with its **Monte Carlo standard error (MCSE)**.

| | |
|---|---|
| **Aims** | Verify coverage, size and power of common procedures used elsewhere in this project |
| **Data-generating mechanisms** | Linear model with homoskedastic vs. heteroskedastic errors; two-sample normal and skewed data |
| **Estimands** | Regression slopes; mean difference; population mean |
| **Methods** | OLS with nonrobust vs. HC3 SEs; pooled vs. Welch t-test; percentile bootstrap |
| **Performance measures** | Bias, empirical SE, model SE, RMSE, coverage, rejection rate (each with MCSE) |

Studies: **(A)** OLS coverage, **(B)** empirical vs. analytic power, **(C)** type-I error of
pooled vs. Welch t-tests, **(D)** bootstrap coverage for a skewed mean.

In [ ]:
# Setup and imports
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import time
from utils.simulation import (
    monte_carlo, performance_summary, simulate_power, rejection_rate,
    dgp_linear, dgp_two_sample, ols_estimator, ttest_estimator,
)
from utils.power import power_ttest
from utils.inference import bootstrap_ci
from utils.reporting import save_table
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = Path("../synthetic_data")
EXPORT_PLOTS = Path("../exports/plots/13_monte_carlo_validation")
EXPORT_TABLES = Path("../exports/tables/13_monte_carlo_validation")
EXPORT_PLOTS.mkdir(parents=True, exist_ok=True)
EXPORT_TABLES.mkdir(parents=True, exist_ok=True)

SEED = 2026

## Study A: OLS coverage under heteroskedasticity

DGP: `y = 1 + 1.0 X1 + 0.5 X2 + e`, n = 60. In the heteroskedastic arm `sd(e) = 1 + |X1|`.
Methods: OLS with conventional (nonrobust) vs. HC3 standard errors. 400 replications.

In [ ]:
N_REPS = 400
truth = {"beta_X1": 1.0, "beta_X2": 0.5}
arms = {}
t0 = time.time()
for err_name, hetero in [("homoskedastic", False), ("heteroskedastic", True)]:
    dgp = dgp_linear(n=60, beta=(1.0, 1.0, 0.5), sigma=1.0, heteroskedastic=hetero)
    for cov in ["nonrobust", "HC3"]:
        res = monte_carlo(dgp, ols_estimator("y ~ X1 + X2", cov_type=cov), n_reps=N_REPS, seed=SEED)
        perf = performance_summary(res, truth)
        perf["errors"] = err_name
        perf["se_type"] = cov
        arms[(err_name, cov)] = perf
print(f"Study A finished in {time.time() - t0:.1f}s")
study_a = pd.concat(arms.values()).reset_index()
cols = ["errors", "se_type", "estimand", "true", "bias", "bias_mcse", "empirical_se", "mean_model_se", "se_ratio", "rmse", "coverage", "coverage_mcse"]
study_a[cols].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sub = study_a[study_a.estimand == "beta_X1"]
labels = sub.errors + "\n" + sub.se_type
ax.bar(labels, sub.coverage, yerr=1.96 * sub.coverage_mcse, capsize=4, color=["C0", "C1", "C0", "C1"])
ax.axhline(0.95, color="red", ls="--", label="nominal 95%")
ax.set_ylim(0.75, 1.0); ax.set_ylabel("coverage of 95% CI for beta_X1")
ax.set_title("Study A: CI coverage (error bars = ±1.96 MCSE)")
ax.legend(); plt.tight_layout()
plt.savefig(EXPORT_PLOTS / "study_a_coverage.png")
plt.show()
save_table(study_a[cols], EXPORT_TABLES / "study_a_ols_coverage", formats=("csv", "md"), index=False)

**Reading the table.** Bias is within a couple of MCSEs of zero in every arm (OLS is unbiased
regardless of heteroskedasticity). Under heteroskedastic errors the *nonrobust* model SE
underestimates the empirical SE (`se_ratio < 1`) and coverage drops below 95%; HC3 restores
`se_ratio ≈ 1` and nominal coverage. The MCSE tells us whether a departure from 0.95 is
larger than simulation noise.

## Study B: Empirical vs. analytic power of the two-sample t-test

n = 30 per group, normal data, sd = 1. Empirical power from 400 replications per effect
size vs. the closed-form power from `utils.power.power_ttest`.

In [ ]:
effects = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
t0 = time.time()
emp = simulate_power(lambda d: dgp_two_sample(30, 30, d, sd=1.0),
                     lambda data: stats.ttest_ind(*data).pvalue,
                     effects=effects, n_reps=400, alpha=0.05, seed=SEED)
emp["analytic_power"] = [power_ttest(effect_size=d, nobs=30, alpha=0.05)["power"] if d > 0 else 0.05 for d in effects]
print(f"Study B finished in {time.time() - t0:.1f}s")
emp

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(emp.effect, emp.power, yerr=1.96 * emp.mcse, fmt="o", capsize=4, label="empirical (±1.96 MCSE)")
grid = np.linspace(0.01, 1.0, 60)
ax.plot(grid, [power_ttest(effect_size=d, nobs=30, alpha=0.05)["power"] for d in grid], color="C3", label="analytic")
ax.axhline(0.05, ls=":", color="gray")
ax.set_xlabel("Cohen's d"); ax.set_ylabel("power")
ax.set_title("Study B: two-sample t-test, n = 30 per group")
ax.legend(); plt.tight_layout()
plt.savefig(EXPORT_PLOTS / "study_b_power_curve.png")
plt.show()
save_table(emp, EXPORT_TABLES / "study_b_power", formats=("csv", "md"), index=False)

## Study C: Type-I error of pooled vs. Welch t-tests

Null hypothesis is true (no mean difference) but variances and sample sizes are unequal:
group 1 has n = 15 and sd = 1, group 2 has n = 45 and sd = 3. The pooled test's size is
distorted when the *smaller* group has the *smaller* variance; Welch's test is robust.

In [ ]:
def dgp_unequal(rng):
    return rng.normal(0, 1.0, 15), rng.normal(0, 3.0, 45)

def dgp_unequal_reversed(rng):
    return rng.normal(0, 3.0, 15), rng.normal(0, 1.0, 45)

rows = []
t0 = time.time()
for scen, dgp in [("small n, small sd", dgp_unequal), ("small n, large sd", dgp_unequal_reversed)]:
    for name, ev in [("pooled t", True), ("Welch t", False)]:
        res = monte_carlo(dgp, ttest_estimator(equal_var=ev), n_reps=500, seed=SEED)
        rate, mcse = rejection_rate(res["p_diff"], alpha=0.05)
        rows.append({"scenario": scen, "test": name, "type_I_error": rate, "mcse": mcse,
                     "within_MC_noise_of_0.05": abs(rate - 0.05) <= 1.96 * mcse})
print(f"Study C finished in {time.time() - t0:.1f}s")
study_c = pd.DataFrame(rows)
save_table(study_c, EXPORT_TABLES / "study_c_type1_error", formats=("csv", "md"), index=False)
study_c

When the small group has the *large* variance the pooled test is anti-conservative
(type-I error well above 5%); when it has the *small* variance it is conservative. Welch's
test holds its size in both scenarios, which is why it should be the default.

## Study D: Bootstrap percentile CI coverage for a skewed mean

Population: LogNormal(0, 1), true mean `exp(0.5) ≈ 1.649`. For n = 20, 50, 200 we draw 200
samples, compute a 95% percentile bootstrap CI (300 resamples) and a t-interval, and record
coverage. The percentile interval is known to undercover for small, skewed samples.

In [ ]:
TRUE_MEAN = float(np.exp(0.5))
rows = []
t0 = time.time()
for n in [20, 50, 200]:
    def dgp(rng, n=n):
        return rng.lognormal(0.0, 1.0, n)

    def est(x):
        b = bootstrap_ci(x, np.mean, n_boot=300, alpha=0.05, method="percentile", seed=int(abs(x[0]) * 1e6) % (2**31))
        t = stats.t.interval(0.95, len(x) - 1, loc=x.mean(), scale=stats.sem(x))
        return {"mean": b.estimate, "mean_ci_low": b.ci_low, "mean_ci_high": b.ci_high,
                "t_mean": x.mean(), "t_mean_ci_low": t[0], "t_mean_ci_high": t[1]}

    res = monte_carlo(dgp, est, n_reps=200, seed=SEED + n)
    perf = performance_summary(res, {"mean": TRUE_MEAN, "t_mean": TRUE_MEAN})
    perf["n"] = n
    perf["method"] = ["percentile bootstrap", "t-interval"]
    rows.append(perf)
print(f"Study D finished in {time.time() - t0:.1f}s")
study_d = pd.concat(rows).reset_index(drop=True)[["n", "method", "true", "mean_estimate", "bias", "coverage", "coverage_mcse"]]
save_table(study_d, EXPORT_TABLES / "study_d_bootstrap_coverage", formats=("csv", "md"), index=False)
study_d

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for m, g in study_d.groupby("method"):
    ax.errorbar(g.n, g.coverage, yerr=1.96 * g.coverage_mcse, fmt="o-", capsize=4, label=m)
ax.axhline(0.95, color="red", ls="--", label="nominal")
ax.set_xscale("log"); ax.set_xticks([20, 50, 200]); ax.set_xticklabels([20, 50, 200])
ax.set_xlabel("sample size n"); ax.set_ylabel("coverage")
ax.set_title("Study D: 95% CI coverage for the mean of LogNormal(0, 1)")
ax.legend(); plt.tight_layout()
plt.savefig(EXPORT_PLOTS / "study_d_bootstrap_coverage.png")
plt.show()

## Take-aways

1. Report MCSE with every simulation result; otherwise differences of a percentage point in
   coverage cannot be distinguished from noise.
2. Nonrobust OLS standard errors fail under heteroskedasticity; HC3 restores nominal coverage.
3. The pooled t-test's size depends on the variance/sample-size pattern; Welch's test does not.
4. Percentile bootstrap intervals undercover for small skewed samples; BCa or larger n helps.

## References

- Morris, T. P., White, I. R., & Crowther, M. J. (2019). Using simulation studies to evaluate statistical methods. *Statistics in Medicine*, 38(11), 2074–2102.
- Long, J. S., & Ervin, L. H. (2000). Using heteroscedasticity consistent standard errors in the linear regression model. *The American Statistician*, 54(3), 217–224.
- Welch, B. L. (1947). The generalization of "Student's" problem when several different population variances are involved. *Biometrika*, 34(1–2), 28–35.
- Delacre, M., Lakens, D., & Leys, C. (2017). Why psychologists should by default use Welch's t-test instead of Student's t-test. *International Review of Social Psychology*, 30(1), 92–101.
- Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.
- Burton, A., Altman, D. G., Royston, P., & Holder, R. L. (2006). The design of simulation studies in medical statistics. *Statistics in Medicine*, 25(24), 4279–4292.